# La búsqueda del techo (v1)

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación de cada familia. Ese escalar lo eligen **mirando
resultados**, así que la cosa que lo elige es un experimento y necesita todo lo
que un experimento necesita — su propia escala, su propio rol del material, su
propia regla de desempate — y hasta ahora no tenía informe propio. Sus seis
celdas vivían adentro del reporte de campaña, que es el lugar donde se leen sus
consecuencias y no donde se juzga cómo se llegó a ellas.

Este cuaderno es ese informe. También corre el **ensayo local**, que es una cosa
distinta de la búsqueda y conviene no confundir:

| | escala | dónde escribe | su respuesta |
| --- | --- | --- | --- |
| **búsqueda** | 20 épocas · 3 semillas | `ceilings.json` | rige la campaña |
| **ensayo** | pocas épocas · 1 semilla | `ceilings.pilot.json` | **no se cita** |

El ensayo contesta *«¿el programa corre?»*, que es lo único que un ensayo puede
contestar. No contesta *«¿cuál es el techo?»*: la rampa avanza con la fracción de
entrenamiento transcurrida, así que a escala corta se satura en la segunda época
y todo techo se alcanza casi enseguida. Lo que mediría es un paisaje donde nada
más entrena.

Por eso son dos archivos y no uno. Con uno solo, el ensayo habría escrito donde
va la respuesta que la campaña consume, y una corrida completa lo habría gastado
sin una palabra.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo."""
    display(Markdown(text))

## 1 · Qué registro rige ahora mismo

Leído del disco, no recordado. El registro completo le gana siempre al ensayo:
un ensayo no desplaza una medición.

In [ ]:
proc = config.ceilings_provenance()
lineas = [f"**registro en vigor:** `{proc['source']}`"]
if proc["record"]:
    lineas.append(f"`{Path(proc['record']).name}` · "
                  f"{proc['epochs']} épocas · {proc['seeds']} semilla(s)")
    if proc.get("atRequiredScale") is False:
        lineas.append("**Está por debajo de la escala que el protocolo declara "
                      f"({proc['requiredScale']}): su techo no se cita.**")
else:
    lineas.append("ninguno todavía — ni la búsqueda ni el ensayo han corrido.")
show(" · ".join(lineas))

## 2 · El ensayo local

Corre el mismo programa que la búsqueda, a su propia escala declarada, y escribe
a su propio archivo. Si ya existe un registro —el que sea— no vuelve a buscar: un
registro que existe significa que la búsqueda contestó, y volver a contestarla
porque un llamador posterior quería otra respuesta es exactamente el silencio que
la negativa de la campaña existe para impedir.

In [ ]:
aforo = config.search_sizing()
bajo, alto = config.CEILING_RANGE
show(
    f"Motor: **{aforo['engine']}** con `GPSampler`.  \n"
    f"La búsqueda real: **{aforo['trials']} trials** por "
    f"`(familia, transferencia)` de **{aforo['epochs']} épocas** — "
    f"{aforo['families']} familias × {aforo['transfers']} transferencias × "
    f"{aforo['trials']} = **{aforo['runs']} corridas**.  \n"
    f"El ensayo: **{config.PILOT_SEARCH_TRIALS} trials** de "
    f"**{config.PILOT_SEARCH_EPOCHS} épocas**, que es "
    f"{config.PILOT_SEARCH_TRIALS * aforo['families'] * aforo['transfers']} "
    f"corridas.  \n"
    f"El rango es continuo, `[{bajo:g}, {alto:g}]` en escala logarítmica, y la "
    f"meseta la define la resolución del criterio sobre el rol de búsqueda: "
    f"**{config.SEARCH_RESOLUTION:g}**, que es una bolsa de las "
    f"{config.VALID_BAGS}."
)

In [ ]:
# Descomentar para correrlo. Escribe `ceilings.pilot.json` y nada más.
# harness.run_search(pilot=True)

## 3 · Lo que la búsqueda eligió

Del registro en vigor. Si es el del ensayo, todo lo de abajo es plumbing y está
dicho arriba.

In [ ]:
show(tables.objective("ceilings"))

In [ ]:
registro = harness.search_record() or harness.search_record(pilot=True)
show(tables.render_ceilings(registro, markdown=True))

In [ ]:
show(tables.conclusion_ceilings(registro))

### 3b · Qué techo rige en cada transferencia

La búsqueda mide unas pocas transferencias y las demás heredan. Esa herencia es
una aplicación **fuera de muestra** y se declara como tal.

In [ ]:
show(tables.objective("ceilings.byTransfer"))

In [ ]:
transferencias = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
show(tables.render_ceilings_by_transfer(registro, transferencias, markdown=True))

In [ ]:
show(tables.conclusion_ceilings_by_transfer(registro, transferencias))

## 4 · Lo que esta búsqueda **no** midió

Es la limitación más grande del escalar que gobierna toda la campaña, y no se lee
en ninguna otra parte. La búsqueda midió unas transferencias y las otras heredaron
su techo sin que nadie lo comprobara ahí.

In [ ]:
medidas = {f"{a}->{b}" for a, b in config.SEARCH_TRANSFERS}
todas = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
heredadas = [t for t in todas if t not in medidas]
show(f"**Medidas:** {', '.join(sorted(medidas))} — "
     f"{len(medidas)} de {len(todas)}.  \n"
     f"**Heredadas:** {', '.join(heredadas)} — "
     f"{len(heredadas)} de {len(todas)}, fuera de muestra.")

## 5 · El sello


In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())